In [ ]:
pip install pandas pyfaidx Bio pyarrow fastparquet

In [1]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"D:/variant_data/dbsnp_unique_for_VEP.parquet"
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host (sẽ được mount vào Docker)
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources" # Nơi chứa các file .bw, .vcf.gz cho plugins

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================
def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        "--fork", "4",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        "--pick",
        "--symbol",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", f"SpliceAI,snv={DOCKER_RESOURCES}/spliceai_scores.raw.snv.ensembl_mane_v1.4.grch38.vcf.gz",
        "--plugin", (
            f"dbNSFP,{DOCKER_RESOURCES}/dbNSFP5.3.1a_grch38.gz," 
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        ),
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,Location,Allele,Gene,SYMBOL,Feature,Feature_type,Consequence,"
            "cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,"
            "MANE_SELECT,CANONICAL,AF,gnomADe_AF,SpliceAI_pred,"
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        subprocess.run(vep_cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[LỖI VEP - Chunk {chunk_idx}]")
        print(e.stderr.strip() or "<empty>")
        raise SystemExit("Dừng pipeline do lỗi chạy VEP.")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu (Sử dụng Uploaded_variation từ VEP và Merge_ID của df)
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")

    if 'SpliceAI_pred' in merged_chunk.columns:
        # Tách chuỗi bằng dấu '|' (sẽ tạo ra list gồm 9 phần tử)
        split_sa = merged_chunk['SpliceAI_pred'].str.split('|', expand=True)
        
        # Lấy đúng 4 cột điểm Delta Score (Vị trí index 1, 2, 3, 4)
        if split_sa.shape[1] == 9:
            merged_chunk['SpliceAI_pred_DS_AG'] = pd.to_numeric(split_sa[1], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_AL'] = pd.to_numeric(split_sa[2], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DG'] = pd.to_numeric(split_sa[3], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DL'] = pd.to_numeric(split_sa[4], errors='coerce')
            
    return merged_chunk

In [2]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

[*] Đang tải dữ liệu từ D:/variant_data/dbsnp_unique_for_VEP.parquet...
[*] Tổng số variant: 201,933. Số lượng chunks: 5


In [3]:
# Định nghĩa file Parquet kết quả cuối cùng
FINAL_OUTPUT_PARQUET = r"D:/variant_data/dbsnp_unique_after_vep.parquet"

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/5 (Rows: 0 - 50000)...
    -> Đang xử lý Chunk 2/5 (Rows: 50000 - 100000)...
    -> Đang xử lý Chunk 3/5 (Rows: 100000 - 150000)...
    -> Đang xử lý Chunk 4/5 (Rows: 150000 - 200000)...
    -> Đang xử lý Chunk 5/5 (Rows: 200000 - 201933)...

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/dbsnp_unique_after_vep.parquet


In [4]:
import pandas as pd
final_df = pd.read_parquet(r"D:/variant_data/dbsnp_unique_after_vep.parquet")

In [5]:
final_df['Consequence'].value_counts()

Consequence
synonymous_variant                                                          80452
missense_variant                                                            72675
stop_gained                                                                 21631
downstream_gene_variant                                                      8539
upstream_gene_variant                                                        5164
missense_variant,splice_region_variant                                       4823
intron_variant                                                               3086
splice_region_variant,synonymous_variant                                     1641
start_lost                                                                   1200
stop_gained,splice_region_variant                                             920
3_prime_UTR_variant                                                           498
splice_polypyrimidine_tract_variant,intron_variant                            309
5_pr

In [6]:
final_df.isnull().sum()

Variant_ID                    0
rsid                          0
CHROM                         0
POS                           0
REF                           0
ALT                           0
gene                          0
clnsig                        0
clnhgvs                       0
clndn                         0
clnrevstat                    0
clnacc                        0
Source                        0
Label                         0
Location                      0
Allele                        0
Gene                          0
SYMBOL                        0
Feature                       0
Feature_type                  0
Consequence                   0
cDNA_position                 0
CDS_position                  0
Protein_position              0
Amino_acids                   0
Codons                        0
MANE_SELECT                   0
CANONICAL                     0
AF                            0
gnomADe_AF                    0
SpliceAI_pred                 0
phyloP10

In [7]:
import numpy as np

# Danh sách giá trị cần kiểm tra
invalid_values = ["-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)
final_df.isnull().sum()

Variant_ID                         0
rsid                               0
CHROM                              0
POS                                0
REF                                0
ALT                                0
gene                               0
clnsig                             0
clnhgvs                            0
clndn                              0
clnrevstat                         0
clnacc                             0
Source                             0
Label                              0
Location                           0
Allele                             0
Gene                              13
SYMBOL                           124
Feature                           11
Feature_type                      11
Consequence                        0
cDNA_position                  17716
CDS_position                   18548
Protein_position               18548
Amino_acids                    18548
Codons                         18548
MANE_SELECT                      849
C

In [8]:
final_df = final_df[final_df['Consequence'] == 'missense_variant']
final_df.isnull().sum()

Variant_ID                        0
rsid                              0
CHROM                             0
POS                               0
REF                               0
ALT                               0
gene                              0
clnsig                            0
clnhgvs                           0
clndn                             0
clnrevstat                        0
clnacc                            0
Source                            0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            8
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                     535
CANONICAL                   

In [10]:
final_df = final_df.dropna(subset=['Consequence', 'SYMBOL', 'MANE_SELECT', 'CANONICAL'])
final_df.isnull().sum()

Variant_ID                        0
rsid                              0
CHROM                             0
POS                               0
REF                               0
ALT                               0
gene                              0
clnsig                            0
clnhgvs                           0
clndn                             0
clnrevstat                        0
clnacc                            0
Source                            0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            0
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                       0
CANONICAL                   

In [11]:
final_df.describe()

,POS,Label
count,7.213300e+04,72133.000000
mean,7.986687e+07,0.492840
std,6.084430e+07,0.499952
min,1.488400e+04,0.000000
25%,3.198417e+07,0.000000
50%,6.235522e+07,0.000000
75%,1.245172e+08,1.000000
max,2.488479e+08,1.000000


In [12]:
final_df['AF'].astype(float).describe()

count    16561.000000
mean         0.029560
std          0.112933
min          0.000000
25%          0.000400
50%          0.001400
75%          0.006200
max          1.000000
Name: AF, dtype: float64

In [13]:
final_df['AF'] = final_df['AF'].astype(float)
final_df['AF'][final_df['AF'] == 0].count()

np.int64(29)

In [14]:
final_df[final_df['AF'].isnull() & final_df['gnomADe_AF'].isnull()][['AF', 'gnomADe_AF']]

,AF,gnomADe_AF
119,NaN,NaN
191,NaN,NaN
321,NaN,NaN
337,NaN,NaN
460,NaN,NaN
...,...,...
201915,NaN,NaN
201916,NaN,NaN
201917,NaN,NaN
201918,NaN,NaN


In [15]:
final_df.dropna(subset=['GERP_92_mammals'], how='all', inplace=True)
final_df.dropna(subset=['phyloP470way_mammalian'], how='all', inplace=True)
final_df.dropna(subset=['GERP++_RS'], how='all', inplace=True)
final_df.dropna(subset=['GERP++_NR'], how='all', inplace=True)

In [16]:
final_df.isnull().sum()

Variant_ID                        0
rsid                              0
CHROM                             0
POS                               0
REF                               0
ALT                               0
gene                              0
clnsig                            0
clnhgvs                           0
clndn                             0
clnrevstat                        0
clnacc                            0
Source                            0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            0
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                       0
CANONICAL                   

In [17]:
final_df.replace({'AF': {np.nan: 0}}, inplace=True)
final_df.replace({'gnomADe_AF': {np.nan: 0}}, inplace=True)

,Variant_ID,rsid,CHROM,POS,REF,ALT,gene,clnsig,clnhgvs,clndn,...,SpliceAI_pred,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
141,1_1014401_G_A,rs138829117,1,1014401,G,A,ISG15:9636,3|3,NC_000001.11:g.1014401G>A,Mendelian_susceptibility_to_mycobacterial_dise...,...,NaN,0.151000,0.636000,-0.126000,0.000000,0.000000,0.087000,2.32,4.21,-1.35
147,1_1022383_C_G,rs768527925,1,1022383,C,G,AGRN:375790,3|3,NC_000001.11:g.1022383C>G,Congenital_myasthenic_syndrome_8|AGRN-related_...,...,NaN,0.783000,0.508000,0.599000,1.000000,0.997000,0.969000,2.3,4.24,-0.022
216,1_1047863_C_T,rs142620337,1,1047863,C,T,AGRN:375790,3|3,NC_000001.11:g.1047863C>T,Congenital_myasthenic_syndrome_8|AGRN-related_...,...,NaN,4.040000,7.453000,0.599000,0.997000,1.000000,0.816000,4.1,4.1,1.33
234,1_1049745_A_G,rs201818235,1,1049745,A,G,AGRN:375790,3,NC_000001.11:g.1049745A>G,Congenital_myasthenic_syndrome_8,...,NaN,3.427000,0.970000,0.688000,1.000000,0.987000,0.175000,3.41,4.55,0.646
248,1_1051275_T_C,rs1557721600,1,1051275,T,C,AGRN:375790,4,NC_000001.11:g.1051275T>C,Congenital_myasthenic_syndrome_8,...,NaN,5.476000,7.718000,0.663000,1.000000,1.000000,0.899000,5.06,5.06,1.47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201780,X_154993134_C_A,rs2073597882,X,154993134,C,A,F8:2157,4,NC_000023.11:g.154993134C>A,Hereditary_factor_VIII_deficiency_disease,...,NaN,6.934000,7.415000,0.599000,1.000000,1.000000,0.962000,5.35,5.35,1.74
201783,X_154996979_A_G,rs1603436638,X,154996979,A,G,F8:2157,4,NC_000023.11:g.154996979A>G,Hereditary_factor_VIII_deficiency_disease,...,NaN,5.766000,10.763000,0.756000,1.000000,1.000000,0.860000,5.03,5.03,1.44
201799,X_155022498_T_G,rs2124174458,X,155022498,T,G,F8:2157,4,NC_000023.11:g.155022498T>G,Hereditary_factor_VIII_deficiency_disease,...,NaN,0.180000,0.015000,0.665000,0.936000,0.003000,0.993000,1.34,5.18,1.6
201806,X_155278006_C_T,rs1032630286,X,155278006,C,T,CLIC2:1193,3,NC_000023.11:g.155278006C>T,not_provided,...,NaN,4.645000,5.286000,0.549000,1.000000,1.000000,0.995000,4.86,4.86,1.85


In [18]:
final_df.isnull().sum()

Variant_ID                        0
rsid                              0
CHROM                             0
POS                               0
REF                               0
ALT                               0
gene                              0
clnsig                            0
clnhgvs                           0
clndn                             0
clnrevstat                        0
clnacc                            0
Source                            0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            0
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                       0
CANONICAL                   

In [19]:
final_df[['gnomADe_AF', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals']] = final_df[['gnomADe_AF', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals']].astype(float)

In [20]:
final_df.describe()

,POS,Label,AF,gnomADe_AF,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
count,1.993000e+04,19930.000000,19930.000000,1.993000e+04,19930.000000,19930.000000,19930.000000,19930.000000,19930.000000,19930.000000,19930.000000,19930.000000,19930.000000
mean,7.957505e+07,0.661515,0.000581,4.849347e-04,5.134894,6.375082,0.499407,0.858320,0.866183,0.787702,3.939465,5.190166,0.980012
std,5.996733e+07,0.473206,0.012872,1.209830e-02,3.320747,4.265047,0.425595,0.325673,0.326520,0.331081,2.532816,0.683917,2.001010
min,4.569500e+04,0.000000,0.000000,0.000000e+00,-7.288000,-20.000000,-4.900000,0.000000,0.000000,0.000000,-12.300000,0.046500,-12.500000
25%,3.343777e+07,0.000000,0.000000,0.000000e+00,2.115250,3.258000,0.592000,0.996000,1.000000,0.720250,3.500000,4.860000,0.358500
50%,6.335034e+07,1.000000,0.000000,6.840000e-07,5.738500,7.591000,0.618000,1.000000,1.000000,0.977000,4.790000,5.340000,1.690000
75%,1.175897e+08,1.000000,0.000000,1.026000e-05,7.902000,9.819000,0.676000,1.000000,1.000000,0.996000,5.460000,5.670000,2.100000
max,2.488479e+08,1.000000,0.833500,7.054000e-01,10.003000,11.936000,0.756000,1.000000,1.000000,1.000000,6.170000,6.170000,8.440000


In [ ]:
final_df.to_parquet(r'D:/variant_data/dbsnp_unique_after_vep_final.parquet', index=False)

In [5]:
df = pd.read_parquet(r'D:/variant_data/dbsnp_unique_after_vep_final.parquet')
df

,Variant_ID,rsid,CHROM,POS,REF,ALT,gene,clnsig,clnhgvs,clndn,...,SpliceAI_pred,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
0,1_1014401_G_A,rs138829117,1,1014401,G,A,ISG15:9636,3|3,NC_000001.11:g.1014401G>A,Mendelian_susceptibility_to_mycobacterial_dise...,...,NaN,0.151,0.636,-0.126,0.000,0.000,0.087,2.32,4.21,-1.350
1,1_1022383_C_G,rs768527925,1,1022383,C,G,AGRN:375790,3|3,NC_000001.11:g.1022383C>G,Congenital_myasthenic_syndrome_8|AGRN-related_...,...,NaN,0.783,0.508,0.599,1.000,0.997,0.969,2.30,4.24,-0.022
2,1_1047863_C_T,rs142620337,1,1047863,C,T,AGRN:375790,3|3,NC_000001.11:g.1047863C>T,Congenital_myasthenic_syndrome_8|AGRN-related_...,...,NaN,4.040,7.453,0.599,0.997,1.000,0.816,4.10,4.10,1.330
3,1_1049745_A_G,rs201818235,1,1049745,A,G,AGRN:375790,3,NC_000001.11:g.1049745A>G,Congenital_myasthenic_syndrome_8,...,NaN,3.427,0.970,0.688,1.000,0.987,0.175,3.41,4.55,0.646
4,1_1051275_T_C,rs1557721600,1,1051275,T,C,AGRN:375790,4,NC_000001.11:g.1051275T>C,Congenital_myasthenic_syndrome_8,...,NaN,5.476,7.718,0.663,1.000,1.000,0.899,5.06,5.06,1.470
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19925,X_154993134_C_A,rs2073597882,X,154993134,C,A,F8:2157,4,NC_000023.11:g.154993134C>A,Hereditary_factor_VIII_deficiency_disease,...,NaN,6.934,7.415,0.599,1.000,1.000,0.962,5.35,5.35,1.740
19926,X_154996979_A_G,rs1603436638,X,154996979,A,G,F8:2157,4,NC_000023.11:g.154996979A>G,Hereditary_factor_VIII_deficiency_disease,...,NaN,5.766,10.763,0.756,1.000,1.000,0.860,5.03,5.03,1.440
19927,X_155022498_T_G,rs2124174458,X,155022498,T,G,F8:2157,4,NC_000023.11:g.155022498T>G,Hereditary_factor_VIII_deficiency_disease,...,NaN,0.180,0.015,0.665,0.936,0.003,0.993,1.34,5.18,1.600
19928,X_155278006_C_T,rs1032630286,X,155278006,C,T,CLIC2:1193,3,NC_000023.11:g.155278006C>T,not_provided,...,NaN,4.645,5.286,0.549,1.000,1.000,0.995,4.86,4.86,1.850


In [1]:
import pandas as pd
import numpy as np
import os

# ============================================================
# CONFIG
# ============================================================

CLINVAR_PATH = r"D:/variant_data/clinvar_vep_mapping_final_with_date.parquet"

# File dbSNP đã chạy VEP
DBSNP_VEP_PATH = r"D:/variant_data/dbsnp_unique_after_vep_final.parquet"

OUTPUT_PATH = r"D:/variant_data/clinvar_dbsnp_vep_merged.parquet"

# ============================================================
# 1. LOAD
# ============================================================

print("=" * 70)
print("LOADING DATA")
print("=" * 70)

clinvar = pd.read_parquet(CLINVAR_PATH)
dbsnp = pd.read_parquet(DBSNP_VEP_PATH)

print(f"ClinVar shape : {clinvar.shape}")
print(f"dbSNP shape   : {dbsnp.shape}")

# ============================================================
# 2. BASIC CHECK
# ============================================================

print("\n" + "=" * 70)
print("CHECKING Variant_ID")
print("=" * 70)

for name, df in [("ClinVar", clinvar), ("dbSNP", dbsnp)]:

    print(f"\n{name}")

    if "Variant_ID" not in df.columns:
        raise ValueError(f"{name} does not contain Variant_ID")

    print("Missing Variant_ID:", df["Variant_ID"].isna().sum())
    print("Duplicate Variant_ID:", df["Variant_ID"].duplicated().sum())
    print("Unique Variant_ID:", df["Variant_ID"].nunique())

    if df["Variant_ID"].duplicated().any():
        raise ValueError(
            f"{name} contains duplicate Variant_ID. "
            "Cannot safely merge one-to-one."
        )

# ============================================================
# 3. FIND COMMON COLUMNS
# ============================================================

clinvar_cols = set(clinvar.columns)
dbsnp_cols = set(dbsnp.columns)

common_cols = sorted(
    (clinvar_cols & dbsnp_cols) - {"Variant_ID"}
)

clinvar_only = sorted(
    clinvar_cols - dbsnp_cols
)

dbsnp_only = sorted(
    dbsnp_cols - clinvar_cols
)

print("\n" + "=" * 70)
print("COLUMN ANALYSIS")
print("=" * 70)

print("\nCommon columns:")
for c in common_cols:
    print("  ", c)

print("\nClinVar-only columns:")
for c in clinvar_only:
    print("  ", c)

print("\ndbSNP-only columns:")
for c in dbsnp_only:
    print("  ", c)

# ============================================================
# 4. CHECK OVERLAP
# ============================================================

print("\n" + "=" * 70)
print("CHECKING OVERLAP")
print("=" * 70)

clinvar_ids = set(clinvar["Variant_ID"])
dbsnp_ids = set(dbsnp["Variant_ID"])

overlap_ids = clinvar_ids & dbsnp_ids

print("ClinVar variants :", len(clinvar_ids))
print("dbSNP variants   :", len(dbsnp_ids))
print("Overlap          :", len(overlap_ids))
print("ClinVar only     :", len(clinvar_ids - dbsnp_ids))
print("dbSNP only       :", len(dbsnp_ids - clinvar_ids))

# ============================================================
# 5. CHECK VALUE CONFLICTS
# ============================================================

print("\n" + "=" * 70)
print("CHECKING COMMON COLUMN CONSISTENCY")
print("=" * 70)

# Chỉ kiểm tra trên các Variant_ID xuất hiện ở cả hai dataset
clinvar_overlap = (
    clinvar[
        clinvar["Variant_ID"].isin(overlap_ids)
    ]
    .set_index("Variant_ID")
)

dbsnp_overlap = (
    dbsnp[
        dbsnp["Variant_ID"].isin(overlap_ids)
    ]
    .set_index("Variant_ID")
)

conflict_summary = []

for col in common_cols:

    # Hai dataframe cùng index Variant_ID
    a = clinvar_overlap[col]
    b = dbsnp_overlap[col]

    # So sánh robust với NaN
    same = (
        a.eq(b)
        | (a.isna() & b.isna())
    )

    conflicts = (~same).sum()

    conflict_summary.append({
        "column": col,
        "conflicts": int(conflicts),
        "overlap_rows": len(overlap_ids)
    })

conflict_df = pd.DataFrame(conflict_summary)

print(conflict_df.to_string(index=False))

# ============================================================
# 6. MERGE
# ============================================================

print("\n" + "=" * 70)
print("MERGING")
print("=" * 70)

# ------------------------------------------------------------
# Rename common columns in dbSNP temporarily
# ------------------------------------------------------------
#
# ClinVar sẽ được ưu tiên cho các cột common.
# dbSNP vẫn được giữ lại nếu ClinVar bị NaN.
#
# Ví dụ:
#
# ClinVar: Consequence
# dbSNP  : Consequence
#
# -> merge thành Consequence
#
# Nhưng trước khi merge:
# ClinVar -> Consequence
# dbSNP   -> Consequence__DBSNP
# ------------------------------------------------------------

dbsnp_temp = dbsnp.copy()

rename_map = {
    col: f"{col}__DBSNP"
    for col in common_cols
}

dbsnp_temp = dbsnp_temp.rename(columns=rename_map)

merged = clinvar.merge(
    dbsnp_temp,
    on="Variant_ID",
    how="outer",
    sort=False,
    validate="one_to_one"
)

print("Initial merged shape:", merged.shape)

# ============================================================
# 7. COALESCE COMMON COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("COALESCING COMMON COLUMNS")
print("=" * 70)

for col in common_cols:

    dbsnp_col = f"{col}__DBSNP"

    if dbsnp_col not in merged.columns:
        continue

    # ClinVar được ưu tiên.
    # Nếu ClinVar missing -> lấy dbSNP.
    merged[col] = merged[col].combine_first(
        merged[dbsnp_col]
    )

    # Xóa cột dbSNP duplicate
    merged.drop(
        columns=[dbsnp_col],
        inplace=True
    )

# ============================================================
# 8. FINAL COLUMN ORDER
# ============================================================

# Variant_ID lên đầu
cols = list(merged.columns)

cols.remove("Variant_ID")
merged = merged[
    ["Variant_ID"] + cols
]

# ============================================================
# 9. CHECK FINAL DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Final shape:", merged.shape)

print(
    "Duplicate Variant_ID:",
    merged["Variant_ID"].duplicated().sum()
)

print(
    "Unique Variant_ID:",
    merged["Variant_ID"].nunique()
)

if merged["Variant_ID"].duplicated().any():
    raise ValueError(
        "FINAL DATASET HAS DUPLICATE Variant_ID!"
    )

# ============================================================
# 10. CHECK CORE VARIANT COLUMNS
# ============================================================

core_cols = [
    "Variant_ID",
    "CHROM",
    "POS",
    "REF",
    "ALT",
    "Label"
]

print("\nCore columns:")

for col in core_cols:
    if col in merged.columns:
        print(
            f"{col:15s} "
            f"dtype={str(merged[col].dtype):10s} "
            f"missing={merged[col].isna().sum()}"
        )
    else:
        print(f"{col:15s} NOT FOUND")

# ============================================================
# 11. CHECK SOURCE
# ============================================================

if "Source" in merged.columns:

    print("\nSource distribution:")
    print(
        merged["Source"]
        .value_counts(dropna=False)
    )

# ============================================================
# 12. SAVE
# ============================================================

print("\n" + "=" * 70)
print("SAVING")
print("=" * 70)

merged.to_parquet(
    OUTPUT_PATH,
    engine="pyarrow",
    index=False
)

print("Saved:")
print(OUTPUT_PATH)

# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"ClinVar rows        : {len(clinvar):,}")
print(f"dbSNP rows          : {len(dbsnp):,}")
print(f"Overlap             : {len(overlap_ids):,}")
print(f"Final merged rows   : {len(merged):,}")
print(f"Final columns       : {len(merged.columns):,}")
print(
    f"Duplicate Variant_ID: "
    f"{merged['Variant_ID'].duplicated().sum():,}"
)

print("\n✓ MERGE COMPLETED SUCCESSFULLY")

LOADING DATA
ClinVar shape : (110020, 55)
dbSNP shape   : (19930, 40)

CHECKING Variant_ID

ClinVar
Missing Variant_ID: 0
Duplicate Variant_ID: 0
Unique Variant_ID: 110020

dbSNP
Missing Variant_ID: 0
Duplicate Variant_ID: 0
Unique Variant_ID: 19930

COLUMN ANALYSIS

Common columns:
   AF
   ALT
   Allele
   Amino_acids
   CANONICAL
   CDS_position
   CHROM
   Codons
   Consequence
   Feature
   Feature_type
   GERP++_NR
   GERP++_RS
   GERP_92_mammals
   Gene
   Label
   Location
   MANE_SELECT
   POS
   Protein_position
   REF
   SYMBOL
   SpliceAI_pred
   cDNA_position
   gnomADe_AF
   phastCons100way_vertebrate
   phastCons17way_primate
   phastCons470way_mammalian
   phyloP100way_vertebrate
   phyloP17way_primate
   phyloP470way_mammalian

ClinVar-only columns:
   #AlleleID
   Assembly
   ChromosomeAccession
   ClinSigSimple
   ClinicalSignificance
   GeneID
   GeneSymbol
   HGNC_ID
   LastEvaluated
   Name
   Oncogenicity
   Origin
   OriginSimple
   PhenotypeIDS
   PhenotypeList

In [2]:
df = pd.read_parquet(r'D:/variant_data/clinvar_dbsnp_vep_merged.parquet')
df.replace({'Source': {np.nan: 'ClinVar'}}, inplace=True)

,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,LastEvaluated,Label,rsid,gene,clnsig,clnhgvs,clndn,clnrevstat,clnacc,Source
0,10_100042515_C_T,4121519.0,single nucleotide variant,NM_001308.3(CPN1):c.1289G>A (p.Ser430Asn),1369.0,CPN1,HGNC:2312,Likely benign,0.0,MedGen:CN169374,...,"May 13, 2025",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
1,10_100065342_C_T,3945266.0,single nucleotide variant,NM_001308.3(CPN1):c.605G>A (p.Arg202Gln),1369.0,CPN1,HGNC:2312,Likely benign,0.0,MedGen:CN169374,...,"Feb 20, 2025",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
2,10_100076062_G_A,2316538.0,single nucleotide variant,NM_001308.3(CPN1):c.269C>T (p.Ala90Val),1369.0,CPN1,HGNC:2312,Likely benign,0.0,MedGen:CN169374,...,"Nov 18, 2022",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
3,10_100209664_T_G,1579390.0,single nucleotide variant,NM_001278.5(CHUK):c.1059A>C (p.Glu353Asp),1147.0,CHUK,HGNC:1974,Likely benign,0.0,MedGen:C3661900|MedGen:CN169374,...,"Feb 06, 2026",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
4,10_100209695_C_T,1929426.0,single nucleotide variant,NM_001278.5(CHUK):c.1028G>A (p.Arg343His),1147.0,CHUK,HGNC:1974,Likely benign,0.0,MedGen:C3661900,...,"Dec 20, 2023",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129945,X_9946907_G_C,3217860.0,single nucleotide variant,NM_001649.4(SHROOM2):c.4821G>C (p.Leu1607Phe),357.0,SHROOM2,HGNC:630,Benign,0.0,NaN,...,"Jul 29, 2024",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
129946,X_9967633_C_G,4117937.0,single nucleotide variant,NM_001195081.2(CLDN34):c.276C>G (p.Phe92Leu),100288814.0,CLDN34,HGNC:51259,Likely benign,0.0,MedGen:CN169374,...,"Apr 11, 2025",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
129947,X_9967697_A_G,2210168.0,single nucleotide variant,NM_001195081.2(CLDN34):c.340A>G (p.Met114Val),100288814.0,CLDN34,HGNC:51259,Likely benign,0.0,MedGen:CN169374,...,"May 25, 2022",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar
129948,X_9967920_C_T,3660319.0,single nucleotide variant,NM_001195081.2(CLDN34):c.563C>T (p.Pro188Leu),100288814.0,CLDN34,HGNC:51259,Likely benign,0.0,MedGen:CN169374,...,"Dec 10, 2024",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ClinVar


In [3]:
df['Source'].value_counts()

Source
ClinVar    110020
dbSNP       19930
Name: count, dtype: int64

In [4]:
df.to_parquet(r'D:/variant_data/clinvar_dbsnp_vep_merged.parquet', index=False)

In [1]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"D:/variant_data/clinvar_dbsnp_vep_merged.parquet"
OUTPUT_PARQUET = r"D:/variant_data/clinvar_vep_final.parquet" # File xuất ra
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources"

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================

def sort_by_chromosome(df):
    """Sắp xếp DataFrame theo đúng thứ tự sinh học: 1->22, X, Y, MT và sắp xếp cả POS."""
    print("Đang sắp xếp dữ liệu theo Chromosome và Position...")
    
    # Chuẩn hóa format CHROM (Xóa chữ 'chr' để dễ map)
    df['CHROM'] = df['CHROM'].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    
    # Tạo từ điển quy đổi Chromosome thành số thứ tự
    chrom_order = {str(i): i for i in range(1, 23)}
    chrom_order.update({'X': 23, 'Y': 24, 'MT': 25, 'M': 25})
    
    # Tạo cột tạm để sắp xếp
    df['CHROM_num'] = df['CHROM'].map(chrom_order)
    
    # Những nhiễm sắc thể lạ (alt contigs, unplaced) sẽ bị đẩy xuống cuối cùng
    df['CHROM_num'] = df['CHROM_num'].fillna(99)
    
    # Sort theo CHROM_num, sau đó tới POS (Vị trí) để chuẩn form VCF nhất
    df = df.sort_values(by=['CHROM_num', 'POS'], ascending=[True, True]).reset_index(drop=True)
    
    # Dọn dẹp cột tạm
    df = df.drop(columns=['CHROM_num'])
    
    print("Hoàn tất sắp xếp!")
    return df

def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "--format", "vcf",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", "GRCh38",
        "--cache",
        "--offline",
        "--fasta",
        f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab",
        "--force_overwrite",
        "--fork", "4",
        # Chọn transcript
        "--mane_select",
        "--canonical",
        "--pick",
        "--symbol",
        # HGVS
        "--hgvs",
        # Protein annotation
        "--protein",
        # Chỉ lấy những field cần thiết
        "--fields",
        (
            "Uploaded_variation,"
            "ENSP,"
            "HGVSc,"
            "HGVSp"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        result = subprocess.run(
            vep_cmd,
            check=True,
            capture_output=True,
            text=True
        )
        print(f"[VEP CHUNK {chunk_idx} THÀNH CÔNG]")

    except subprocess.CalledProcessError as e:
        print("\n" + "=" * 70)
        print(f"[VEP ERROR - CHUNK {chunk_idx}]")
        print("=" * 70)
        print("\nReturn code:")
        print(e.returncode)
        print("\nSTDERR:")
        print(e.stderr if e.stderr else "<empty>")
        raise SystemExit(f"Dừng pipeline do VEP failed at chunk {chunk_idx}")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")

    if 'SpliceAI_pred' in merged_chunk.columns:
        # Tách chuỗi bằng dấu '|' 
        split_sa = merged_chunk['SpliceAI_pred'].str.split('|', expand=True)
        # Lấy đúng 4 cột điểm Delta Score
        if split_sa.shape[1] == 9:
            merged_chunk['SpliceAI_pred_DS_AG'] = pd.to_numeric(split_sa[1], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_AL'] = pd.to_numeric(split_sa[2], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DG'] = pd.to_numeric(split_sa[3], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DL'] = pd.to_numeric(split_sa[4], errors='coerce')
            
    return merged_chunk

In [2]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
df_full = sort_by_chromosome(df_full)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

[*] Đang tải dữ liệu từ D:/variant_data/clinvar_dbsnp_vep_merged.parquet...
Đang sắp xếp dữ liệu theo Chromosome và Position...
Hoàn tất sắp xếp!
[*] Tổng số variant: 129,950. Số lượng chunks: 3


In [3]:
# Định nghĩa file Parquet kết quả cuối cùng
FINAL_OUTPUT_PARQUET = r"D:/variant_data/clinvar_dbsnp_vep_merged_final.parquet"

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/3 (Rows: 0 - 50000)...
[VEP CHUNK 1 THÀNH CÔNG]
    -> Đang xử lý Chunk 2/3 (Rows: 50000 - 100000)...
[VEP CHUNK 2 THÀNH CÔNG]
    -> Đang xử lý Chunk 3/3 (Rows: 100000 - 129950)...
[VEP CHUNK 3 THÀNH CÔNG]

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/clinvar_dbsnp_vep_merged_final.parquet


In [5]:
df = pd.read_parquet(r'D:/variant_data/clinvar_dbsnp_vep_merged_final.parquet')
df.isnull().sum()

Variant_ID         0
#AlleleID      19930
Type           19930
Name           19930
GeneID         19930
               ...  
clnacc        110020
Source             0
ENSP               0
HGVSc              0
HGVSp              0
Length: 66, dtype: int64